In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-huggingface sentence-transformers langchain-community

from dotenv import load_dotenv
load_dotenv()

import os
from langchain_huggingface import HuggingFaceEmbeddings

# FREE embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded!")

Note: you may need to restart the kernel to use updated packages.


/Users/hemanth/Desktop/sem 6/GenAI/hands-on_unit2/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/hemanth/Desktop/sem 6/GenAI/hands-on_unit2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding model loaded!


In [2]:
vector = embeddings.embed_query("Apple")

print("Dimensionality:", len(vector))
print("First 5 numbers:", vector[:5])

Dimensionality: 384
First 5 numbers: [-0.006138534285128117, 0.03101177327334881, 0.06479363143444061, 0.010941469110548496, 0.005267174914479256]


In [3]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

vec_cat = embeddings.embed_query("Cat")
vec_dog = embeddings.embed_query("Dog")
vec_car = embeddings.embed_query("Car")

print("Cat vs Dog:", cosine_similarity(vec_cat, vec_dog))
print("Cat vs Car:", cosine_similarity(vec_cat, vec_car))

Cat vs Dog: 0.6606375808683055
Cat vs Car: 0.4633276137538393


part4b

In [10]:
%pip install python-dotenv --upgrade --quiet faiss-cpu langchain-huggingface sentence-transformers langchain-community

from dotenv import load_dotenv
load_dotenv()

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("RAG LLM ready!")

Note: you may need to restart the kernel to use updated packages.
RAG LLM ready!


In [12]:
# Force reload the environment variables
import importlib
import dotenv
importlib.reload(dotenv)

from dotenv import load_dotenv
load_dotenv(override=True)

# Recreate the LLM with new API key
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

print("LLM recreated with new API key!")

LLM recreated with new API key!


In [5]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Piyush's favorite food is Pizza with extra cheese."),
    Document(page_content="The secret password to the lab is 'Blueberry'."),
    Document(page_content="LangChain is a framework for developing AI applications."),
]

print("Knowledge base created!")

Knowledge base created!


In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

print("FAISS vector store created.")

FAISS vector store created.


In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """
Answer based ONLY on the context below:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What is the secret password?"))

The secret password is 'Blueberry'.


part4c

In [1]:
import faiss
import numpy as np

# Mock dataset: 10,000 vectors of size 128
d = 128
nb = 10000
xb = np.random.random((nb, d)).astype('float32')

In [2]:
index = faiss.IndexFlatL2(d)
index.add(xb)

print("Flat Index contains:", index.ntotal, "vectors")

Flat Index contains: 10000 vectors


In [3]:
nlist = 100  # number of clusters
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)

index_ivf.train(xb)
index_ivf.add(xb)

print("IVF Index built with", nlist, "clusters.")

IVF Index built with 100 clusters.


In [5]:
M = 16
index_hnsw = faiss.IndexHNSWFlat(d, M)
index_hnsw.add(xb)

print("HNSW Index created.")

HNSW Index created.
